In [2]:
# ============================================================
# SMS SPAM ENSEMBLE CLASSIFICATION (GOOGLE COLAB FINAL)
# Using dataset path: /content/SMSSpamCollection
# ============================================================

# ---------------- INSTALL ----------------
!pip install -q scikit-learn pandas numpy

# ---------------- IMPORTS ----------------
import numpy as np
import pandas as pd
import re

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import VotingClassifier, StackingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.calibration import CalibratedClassifierCV

from google.colab import files

# ---------------- LOAD DATA ----------------
data_path = "/content/SMSSpamCollection"

df = pd.read_csv(
    data_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

# save csv version (optional)
df.to_csv("/content/sms.csv", index=False)

# ---------------- CLEAN TEXT ----------------
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

df["message"] = df["message"].astype(str).apply(clean_text)
df["label"] = df["label"].map({"ham": 0, "spam": 1})

X_text = df["message"].values
y = df["label"].values

# ---------------- TF-IDF ----------------
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
X = vectorizer.fit_transform(X_text)

# ---------------- MODELS ----------------
nb = MultinomialNB()
lr = LogisticRegression(max_iter=1000)
svm = CalibratedClassifierCV(LinearSVC())

voting_hard = VotingClassifier(
    estimators=[("nb", nb), ("lr", lr), ("svm", svm)],
    voting="hard"
)

voting_soft = VotingClassifier(
    estimators=[("nb", nb), ("lr", lr), ("svm", svm)],
    voting="soft"
)

stacking = StackingClassifier(
    estimators=[("nb", nb), ("lr", lr), ("svm", svm)],
    final_estimator=LogisticRegression()
)

# AdaBoost with Decision Stumps (depth = 1)
stump = DecisionTreeClassifier(max_depth=1)

adaboost = AdaBoostClassifier(
    estimator=stump,
    n_estimators=100,
    random_state=42
)

models = {
    "NaiveBayes": nb,
    "LogisticRegression": lr,
    "LinearSVM": svm,
    "VotingHard": voting_hard,
    "VotingSoft": voting_soft,
    "Stacking": stacking,
    "AdaBoost_Stumps": adaboost
}

# ---------------- EVALUATION FUNCTION ----------------
def evaluate_model(model, X, y, k=5):

    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    precision_scores, recall_scores, f1_scores, roc_scores = [], [], [], []
    all_true, all_pred = [], []

    for train_idx, test_idx in skf.split(X, y):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)[:, 1]
        else:
            y_prob = y_pred

        precision_scores.append(precision_score(y_test, y_pred))
        recall_scores.append(recall_score(y_test, y_pred))
        f1_scores.append(f1_score(y_test, y_pred))
        roc_scores.append(roc_auc_score(y_test, y_prob))

        all_true.extend(y_test)
        all_pred.extend(y_pred)

    cm = confusion_matrix(all_true, all_pred)

    return {
        "Precision": f"{np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}",
        "Recall": f"{np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}",
        "F1": f"{np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
        "ROC_AUC": f"{np.mean(roc_scores):.4f} ± {np.std(roc_scores):.4f}",
        "ConfusionMatrix": cm
    }

# ---------------- TRAIN & COMPARE ----------------
results = []

for name, model in models.items():
    print(f"\nEvaluating {name}...")

    res = evaluate_model(model, X, y, k=5)

    print("Confusion Matrix:\n", res["ConfusionMatrix"])

    results.append({
        "Model": name,
        "Precision": res["Precision"],
        "Recall": res["Recall"],
        "F1": res["F1"],
        "ROC_AUC": res["ROC_AUC"]
    })

results_df = pd.DataFrame(results)
results_df.to_csv("/content/ensemble_comparison.csv", index=False)

print("\nModel Comparison:")
display(results_df)

# ---------------- FINAL MODEL ----------------
best_model = voting_soft
best_model.fit(X, y)

probs = best_model.predict_proba(X)[:, 1]
preds = best_model.predict(X)

final_df = pd.DataFrame({
    "MessageId": np.arange(len(y)),
    "Actual": y,
    "Predicted": preds,
    "Probability": probs
})

final_df.to_csv("/content/final_model_predictions.csv", index=False)

print("\n✅ Files Generated Successfully!")

# Download results
files.download("/content/ensemble_comparison.csv")
files.download("/content/final_model_predictions.csv")


Evaluating NaiveBayes...
Confusion Matrix:
 [[4822    3]
 [ 140  607]]

Evaluating LogisticRegression...
Confusion Matrix:
 [[4820    5]
 [ 243  504]]

Evaluating LinearSVM...
Confusion Matrix:
 [[4801   24]
 [  73  674]]

Evaluating VotingHard...
Confusion Matrix:
 [[4818    7]
 [ 124  623]]

Evaluating VotingSoft...
Confusion Matrix:
 [[4818    7]
 [ 118  629]]

Evaluating Stacking...
Confusion Matrix:
 [[4810   15]
 [  65  682]]

Evaluating AdaBoost_Stumps...
Confusion Matrix:
 [[4811   14]
 [ 445  302]]

Model Comparison:


,Model,Precision,Recall,F1,ROC_AUC
0,NaiveBayes,0.9952 ± 0.0064,0.8125 ± 0.0394,0.8941 ± 0.0236,0.9859 ± 0.0044
1,LogisticRegression,0.9902 ± 0.0003,0.6747 ± 0.0187,0.8024 ± 0.0134,0.9882 ± 0.0061
2,LinearSVM,0.9658 ± 0.0100,0.9022 ± 0.0251,0.9327 ± 0.0128,0.9902 ± 0.0047
3,VotingHard,0.9889 ± 0.0037,0.8340 ± 0.0384,0.9044 ± 0.0230,0.9163 ± 0.0191
4,VotingSoft,0.9890 ± 0.0036,0.8420 ± 0.0353,0.9092 ± 0.0209,0.9904 ± 0.0044
5,Stacking,0.9786 ± 0.0101,0.9129 ± 0.0226,0.9444 ± 0.0127,0.9901 ± 0.0040
6,AdaBoost_Stumps,0.9555 ± 0.0126,0.4043 ± 0.0133,0.5681 ± 0.0149,0.9158 ± 0.0124



✅ Files Generated Successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>